# ITEM 업서트 노트북

이 노트북은 `ITEM_업로드 데이터.xlsx`를 읽어서 `modular.db`의 `ITEM` 테이블에 업서트합니다.

핵심 규칙:
- 엑셀의 `condition.id`를 기준으로 item id prefix를 자동 생성합니다.
- 예: `STS_ADULT` -> `STS_ADULT_ITEM_1`, `STS_ADULT_ITEM_2`, ...
- 각 row의 `test.id`, `condition.id`, `template_id`, `choice.id`가 실제 FK 대상 테이블에 존재하는지 먼저 검증한 뒤 업서트합니다.

실행 전 준비:
1. `ITEMCONDITION` 데이터가 먼저 들어가 있어야 합니다.
2. `TEMPLATE` 데이터가 먼저 들어가 있어야 합니다.
3. `CHOICE` 데이터가 먼저 들어가 있어야 합니다.
4. 엑셀의 `choice.id` 컬럼을 실제 DB 값에 맞게 채웁니다.


## 엑셀 업로드 포맷

엑셀 첫 행 헤더는 아래 컬럼을 사용합니다.

- `no`
- `text`
- `template_id`
- `meta_json`
- `condition.id`
- `test.id`
- `choice.id`

`id`는 노트북에서 자동 생성하고, 나머지 FK 컬럼은 엑셀에서 직접 입력합니다.


In [5]:
# 1) 경로/스키마 설정
from pathlib import Path
import json
import re
import sqlite3
import zipfile
import xml.etree.ElementTree as ET

DB_PATH = Path(r"C:\Users\user\workspace\2.0-modular\modular.db")
XLSX_PATH = Path(r"C:\Users\user\workspace\2.0-modular\app\db\mig\ITEM_업로드_데이터.xlsx")
TABLE_NAME = 'ITEM'

SCHEMA = [
    {'name': 'id', 'type': 'TEXT', 'constraints': 'NOT NULL'},
    {'name': 'no', 'type': 'TEXT', 'constraints': 'NOT NULL'},
    {'name': 'text', 'type': 'TEXT', 'constraints': ''},
    {'name': 'template_id', 'type': 'TEXT', 'constraints': 'NOT NULL'},
    {'name': 'meta_json', 'type': 'TEXT', 'constraints': ''},
    {'name': 'condition.id', 'type': 'TEXT', 'constraints': 'NOT NULL'},
    {'name': 'test.id', 'type': 'TEXT', 'constraints': 'NOT NULL'},
    {'name': 'choice.id', 'type': 'TEXT', 'constraints': 'NOT NULL'},
]
PRIMARY_KEY = ['id', 'test.id', 'condition.id', 'template_id', 'choice.id']

VALID_TYPES = {'INTEGER', 'REAL', 'TEXT', 'BLOB'}
identifier_re = re.compile(r'^[A-Za-z_][A-Za-z0-9_\.]*$')

def validate_identifier(name: str) -> None:
    if not identifier_re.match(name):
        raise ValueError(f'잘못된 이름: {name} (영문/숫자/언더스코어/점만 허용, 숫자로 시작 불가)')

validate_identifier(TABLE_NAME)
seen = set()
for col in SCHEMA:
    col_name = col['name']
    col_type = col['type'].upper()
    validate_identifier(col_name)
    if col_name in seen:
        raise ValueError(f'중복 컬럼명: {col_name}')
    if col_type not in VALID_TYPES:
        raise ValueError(f'지원하지 않는 타입: {col_type}')
    seen.add(col_name)

print('설정 확인 완료')
print('DB_PATH =', DB_PATH)
print('XLSX_PATH =', XLSX_PATH)
print('TABLE_NAME =', TABLE_NAME)
print('PRIMARY_KEY =', PRIMARY_KEY)


설정 확인 완료
DB_PATH = C:\Users\user\workspace\2.0-modular\modular.db
XLSX_PATH = C:\Users\user\workspace\2.0-modular\app\db\mig\ITEM_업로드_데이터.xlsx
TABLE_NAME = ITEM
PRIMARY_KEY = ['id', 'test.id', 'condition.id', 'template_id', 'choice.id']


In [6]:
# 2) ITEM 테이블 구조 확인
conn = sqlite3.connect(DB_PATH)
try:
    cur = conn.execute('PRAGMA table_info("ITEM")')
    print('table_info(ITEM)')
    for row in cur.fetchall():
        print(row)

    print('\nforeign_key_list(ITEM)')
    cur = conn.execute('PRAGMA foreign_key_list("ITEM")')
    for row in cur.fetchall():
        print(row)
finally:
    conn.close()


table_info(ITEM)
(0, 'id', 'TEXT', 1, None, 1)
(1, 'no', 'TEXT', 1, None, 0)
(2, 'text', 'TEXT', 0, None, 0)
(3, 'template_id', 'TEXT', 1, None, 4)
(4, 'meta_json', 'TEXT', 0, None, 0)
(5, 'condition.id', 'TEXT', 1, None, 3)
(6, 'test.id', 'TEXT', 1, None, 2)
(7, 'choice.id', 'TEXT', 1, None, 5)

foreign_key_list(ITEM)
(0, 0, 'CHOICE', 'choice.id', 'id', 'NO ACTION', 'NO ACTION', 'NONE')
(1, 0, 'ITEMCONDITION', 'test.id', 'test.id', 'NO ACTION', 'NO ACTION', 'NONE')
(1, 1, 'ITEMCONDITION', 'condition.id', 'id', 'NO ACTION', 'NO ACTION', 'NONE')
(2, 0, 'TEMPLATE', 'template_id', 'id', 'NO ACTION', 'NO ACTION', 'NONE')


In [7]:
# 3) 엑셀 미리보기
NS = {'x': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}

def read_xlsx_rows(xlsx_path: Path) -> list[dict]:
    with zipfile.ZipFile(xlsx_path) as z:
        shared_strings = []
        shared_root = ET.fromstring(z.read('xl/sharedStrings.xml'))
        for si in shared_root.findall('x:si', NS):
            texts = [t.text or '' for t in si.findall('.//x:t', NS)]
            shared_strings.append(''.join(texts))

        sheet_root = ET.fromstring(z.read('xl/worksheets/sheet1.xml'))
        rows = []
        for row in sheet_root.findall('.//x:sheetData/x:row', NS):
            row_map = {}
            for cell in row.findall('x:c', NS):
                ref = cell.attrib.get('r', '')
                col = ''.join(ch for ch in ref if ch.isalpha())
                cell_type = cell.attrib.get('t')
                value_node = cell.find('x:v', NS)
                value = '' if value_node is None else (value_node.text or '')
                if cell_type == 's' and value != '':
                    value = shared_strings[int(value)]
                row_map[col] = value
            rows.append(row_map)

    if not rows:
        return []

    headers = rows[0]
    data_rows = []
    for raw in rows[1:]:
        mapped = {headers.get(col, col): val for col, val in raw.items()}
        data_rows.append(mapped)
    return data_rows

xlsx_rows = read_xlsx_rows(XLSX_PATH)
print('엑셀 row 수 =', len(xlsx_rows))
for row in xlsx_rows[:10]:
    print(row)


엑셀 row 수 = 240
{'no': '1', 'text': '처음 만난 사람들과 있게 되면, 나는', 'template_id': 'TPL_BIPOLAR_WITH_PROMPT', 'condition.id': 'GOLDEN_ADULT', 'test.id': 'GOLDEN', 'choice.id': 'CHOICE_GOLDEN_ADULT_PART1_BIPOLAR_7'}
{'no': '2', 'text': '나는 주로', 'template_id': 'TPL_BIPOLAR_WITH_PROMPT', 'condition.id': 'GOLDEN_ADULT', 'test.id': 'GOLDEN', 'choice.id': 'CHOICE_GOLDEN_ADULT_PART1_BIPOLAR_7'}
{'no': '3', 'text': '휴가 중에 내가 더 많은 시간을 쓰고 싶은 것은', 'template_id': 'TPL_BIPOLAR_WITH_PROMPT', 'condition.id': 'GOLDEN_ADULT', 'test.id': 'GOLDEN', 'choice.id': 'CHOICE_GOLDEN_ADULT_PART1_BIPOLAR_7'}
{'no': '4', 'text': '게임이나 스포츠를 처음 시작할 때, 나는', 'template_id': 'TPL_BIPOLAR_WITH_PROMPT', 'condition.id': 'GOLDEN_ADULT', 'test.id': 'GOLDEN', 'choice.id': 'CHOICE_GOLDEN_ADULT_PART1_BIPOLAR_7'}
{'no': '5', 'text': '나만의 정원을 꾸민다면, 내가 좋아하는 정원은', 'template_id': 'TPL_BIPOLAR_WITH_PROMPT', 'condition.id': 'GOLDEN_ADULT', 'test.id': 'GOLDEN', 'choice.id': 'CHOICE_GOLDEN_ADULT_PART1_BIPOLAR_7'}
{'no': '6', 'text': '모임에서 떠나려 할 

## 업서트


In [8]:
# 4) ITEM 테이블 업서트
insert_columns = [col['name'] for col in SCHEMA]
schema_type_map = {col['name']: col['type'].upper() for col in SCHEMA}
print('입력 대상 컬럼:', insert_columns)

def normalize_value(col_name: str, value):
    col_type = schema_type_map[col_name]
    if value == '':
        return None
    if col_type == 'TEXT' and isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return value

def parse_meta_json(raw_value):
    if raw_value in (None, ''):
        return None
    text = str(raw_value).strip()
    if not text:
        return None
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return text

def normalize_id_token(value: str) -> str:
    token = re.sub(r'[^A-Za-z0-9]+', '_', str(value).strip().upper())
    return token.strip('_')

def build_item_prefix(test_id: str, condition_id: str) -> str:
    cond_token = normalize_id_token(condition_id)
    if not cond_token:
        test_token = normalize_id_token(test_id)
        return f'{test_token}_ITEM_' if test_token else 'ITEM_'
    return f'{cond_token}_ITEM_'

def build_item_id(test_id: str, condition_id: str, seq: int) -> str:
    return f'{build_item_prefix(test_id, condition_id)}{seq}'

materialized_rows = []
sequence_by_group = {}
required_excel_keys = {'no', 'template_id', 'condition.id', 'test.id', 'choice.id'}

conn = sqlite3.connect(DB_PATH)
try:
    conn.execute('PRAGMA foreign_keys = ON;')
    existing_test_ids = {row[0] for row in conn.execute('SELECT "id" FROM "TEST"')}
    existing_template_ids = {row[0] for row in conn.execute('SELECT "id" FROM "TEMPLATE"')}
    existing_choice_ids = {row[0] for row in conn.execute('SELECT "id" FROM "CHOICE"')}
    existing_itemconditions = {
        (row[0], row[1])
        for row in conn.execute('SELECT "id", "test.id" FROM "ITEMCONDITION"')
    }

    for i, row in enumerate(xlsx_rows, start=1):
        row_keys = set(row.keys())
        missing = [k for k in required_excel_keys if k not in row_keys]
        if missing:
            raise ValueError(f'{i}번째 엑셀 row에 필수 컬럼 누락: {missing}')

        test_id = str(row.get('test.id', '')).strip()
        condition_id = str(row.get('condition.id', '')).strip()
        template_id = str(row.get('template_id', '')).strip()
        choice_id = str(row.get('choice.id', '')).strip()
        item_no = str(row.get('no', '')).strip()
        if not test_id or not condition_id or not template_id or not choice_id or not item_no:
            raise ValueError(f'{i}번째 엑셀 row에 필수값이 비어 있습니다: {row}')

        if test_id not in existing_test_ids:
            raise KeyError(f'{i}번째 row의 test.id={test_id} 가 TEST 테이블에 없습니다.')
        if (condition_id, test_id) not in existing_itemconditions:
            raise KeyError(
                f'{i}번째 row의 (condition.id={condition_id}, test.id={test_id}) 가 ITEMCONDITION 테이블에 없습니다.'
            )
        if template_id not in existing_template_ids:
            raise KeyError(f'{i}번째 row의 template_id={template_id} 가 TEMPLATE 테이블에 없습니다.')
        if choice_id not in existing_choice_ids:
            raise KeyError(f'{i}번째 row의 choice.id={choice_id} 가 CHOICE 테이블에 없습니다.')

        group_key = (test_id, condition_id)
        sequence_by_group[group_key] = sequence_by_group.get(group_key, 0) + 1
        seq = sequence_by_group[group_key]

        meta_value = parse_meta_json(row.get('meta_json'))
        materialized = {
            'id': build_item_id(test_id, condition_id, seq),
            'no': item_no,
            'text': row.get('text'),
            'template_id': template_id,
            'meta_json': meta_value,
            'condition.id': condition_id,
            'test.id': test_id,
            'choice.id': choice_id,
        }
        normalized = {k: normalize_value(k, materialized[k]) for k in insert_columns}
        materialized_rows.append(normalized)

    placeholders = ', '.join(['?'] * len(insert_columns))
    col_sql = ', '.join([f'"{c}"' for c in insert_columns])
    conflict_columns = ['id', 'test.id', 'condition.id', 'template_id', 'choice.id']
    update_columns = [c for c in insert_columns if c not in conflict_columns]
    conflict_sql = ', '.join([f'"{c}"' for c in conflict_columns])
    update_sql = ', '.join([f'"{c}" = excluded."{c}"' for c in update_columns])
    insert_sql = (
        f'INSERT INTO "{TABLE_NAME}" ({col_sql}) VALUES ({placeholders}) '
        f'ON CONFLICT ({conflict_sql}) DO UPDATE SET {update_sql}'
    )
    values = [tuple(row[c] for c in insert_columns) for row in materialized_rows]

    print('실행 SQL:', insert_sql)
    print('업서트 대상 건수:', len(values))
    print('샘플 생성 id:', [row['id'] for row in materialized_rows[:10]])

    conn.executemany(insert_sql, values)
    conn.commit()
    print(f'업서트 완료: {len(values)}건')
finally:
    conn.close()


입력 대상 컬럼: ['id', 'no', 'text', 'template_id', 'meta_json', 'condition.id', 'test.id', 'choice.id']
실행 SQL: INSERT INTO "ITEM" ("id", "no", "text", "template_id", "meta_json", "condition.id", "test.id", "choice.id") VALUES (?, ?, ?, ?, ?, ?, ?, ?) ON CONFLICT ("id", "test.id", "condition.id", "template_id", "choice.id") DO UPDATE SET "no" = excluded."no", "text" = excluded."text", "meta_json" = excluded."meta_json"
업서트 대상 건수: 240
샘플 생성 id: ['GOLDEN_ADULT_ITEM_1', 'GOLDEN_ADULT_ITEM_2', 'GOLDEN_ADULT_ITEM_3', 'GOLDEN_ADULT_ITEM_4', 'GOLDEN_ADULT_ITEM_5', 'GOLDEN_ADULT_ITEM_6', 'GOLDEN_ADULT_ITEM_7', 'GOLDEN_ADULT_ITEM_8', 'GOLDEN_ADULT_ITEM_9', 'GOLDEN_ADULT_ITEM_10']
업서트 완료: 240건


In [9]:
# 5) 결과 조회
conn = sqlite3.connect(DB_PATH)
try:
    cur = conn.execute(
        'SELECT "id", "no", "text", "template_id", "condition.id", "test.id", "choice.id" FROM "ITEM" ORDER BY "test.id", "condition.id", CAST("no" AS INTEGER)'
    )
    rows = cur.fetchall()
    print(f'총 {len(rows)}건')
    for row in rows[:40]:
        print(row)
    if len(rows) > 40:
        print('...')
finally:
    conn.close()


총 368건
('GOLDEN_ADULT_ITEM_1', '1', '처음 만난 사람들과 있게 되면, 나는', 'TPL_BIPOLAR_WITH_PROMPT', 'GOLDEN_ADULT', 'GOLDEN', 'CHOICE_GOLDEN_ADULT_PART1_BIPOLAR_7')
('GOLDEN_ADULT_ITEM_2', '2', '나는 주로', 'TPL_BIPOLAR_WITH_PROMPT', 'GOLDEN_ADULT', 'GOLDEN', 'CHOICE_GOLDEN_ADULT_PART1_BIPOLAR_7')
('GOLDEN_ADULT_ITEM_3', '3', '휴가 중에 내가 더 많은 시간을 쓰고 싶은 것은', 'TPL_BIPOLAR_WITH_PROMPT', 'GOLDEN_ADULT', 'GOLDEN', 'CHOICE_GOLDEN_ADULT_PART1_BIPOLAR_7')
('GOLDEN_ADULT_ITEM_4', '4', '게임이나 스포츠를 처음 시작할 때, 나는', 'TPL_BIPOLAR_WITH_PROMPT', 'GOLDEN_ADULT', 'GOLDEN', 'CHOICE_GOLDEN_ADULT_PART1_BIPOLAR_7')
('GOLDEN_ADULT_ITEM_5', '5', '나만의 정원을 꾸민다면, 내가 좋아하는 정원은', 'TPL_BIPOLAR_WITH_PROMPT', 'GOLDEN_ADULT', 'GOLDEN', 'CHOICE_GOLDEN_ADULT_PART1_BIPOLAR_7')
('GOLDEN_ADULT_ITEM_6', '6', '모임에서 떠나려 할 때, 나는', 'TPL_BIPOLAR_WITH_PROMPT', 'GOLDEN_ADULT', 'GOLDEN', 'CHOICE_GOLDEN_ADULT_PART1_BIPOLAR_7')
('GOLDEN_ADULT_ITEM_7', '7', '프로젝트를 시작하기 전에, 나는', 'TPL_BIPOLAR_WITH_PROMPT', 'GOLDEN_ADULT', 'GOLDEN', 'CHOICE_GOLDEN_ADULT_PART1